# 中国货币政策报告文本特征与金融市场反应

本 Notebook 展示锁定分析计划、正式样本、章节修复、文本创新度、事件窗口、股票波动、股票收益和收益率曲线的核心计算。

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
while not (ROOT / 'configs/project.yml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))
print(ROOT)

D:\PyCharm\Quant\monetary_policy_project


## 1. 锁定分析计划和正式样本

In [2]:
from src.monetary_policy.sample import verify_final_analysis_plan, sample_bounds, is_in_formal_sample
verify_final_analysis_plan()
print('formal sample:', sample_bounds())

formal sample: ('2006Q1', '2025Q4')


## 2. 数据来源和覆盖范围

In [3]:
registry = pd.read_csv(ROOT / 'data/source_registry.csv')
registry[['dataset_name','source_organization','coverage_start','coverage_end','license_or_terms']].head()

,dataset_name,source_organization,coverage_start,coverage_end,license_or_terms
0,pbc_report_2006Q1,中国人民银行,2006Q1,2006Q1,PBC public website; redistribution terms not s...
1,pbc_report_2006Q2,中国人民银行,2006Q2,2006Q2,PBC public website; redistribution terms not s...
2,pbc_report_2006Q3,中国人民银行,2006Q3,2006Q3,PBC public website; redistribution terms not s...
3,pbc_report_2006Q4,中国人民银行,2006Q4,2006Q4,PBC public website; redistribution terms not s...
4,pbc_report_2007Q1,中国人民银行,2007Q1,2007Q1,PBC public website; redistribution terms not s...


In [4]:
meta = pd.read_csv(ROOT / 'data/processed/pbc_report_metadata.csv')
meta['in_formal_sample'] = meta['report_period'].map(is_in_formal_sample)
meta.groupby('in_formal_sample').size()

in_formal_sample
False     1
True     80
dtype: int64

## 3. 原始 PDF 抽取文本样例

In [5]:
from src.monetary_policy.data.pbc_reports import report_text_path
sample_id = meta.loc[meta['in_formal_sample'], 'report_id'].iloc[-1]
sample_path = report_text_path(sample_id)
raw_text = sample_path.read_text(encoding='utf-8')
raw_text[:800]

'中国货币政策执行报告\n2025 年第四季度\n中国人民银行货币政策分析小组\n2026 年2 月10 日\n\nI\n内容摘要\n2025 年是“十四五”收官之年，在以习近平同志为核心的党中\n央坚强领导下，国民经济延续稳中有进发展态势，经济社会发展主要\n目标顺利实现。全年国内生产总值（GDP）同比增长5%。中国人民银\n行坚持以习近平新时代中国特色社会主义思想为指导，坚决落实党中\n央、国务院决策部署，实施适度宽松的货币政策，在执行好存量货币\n政策的基础上，又推出一揽子货币金融政策组合，强化逆周期调节，\n有效支持实体经济稳定增长和金融市场平稳运行。\n一是保持货币信贷合理增长。综合运用存款准备金率、公开市场\n操作等多种货币政策工具，保持流动性充裕。引导金融机构加强项目\n储备和信贷投放，充分满足实体经济有效信贷需求。二是推动社会综\n合融资成本低位下行。下调政策利率、结构性货币政策工具利率和个\n人住房公积金贷款利率，有力支持降低社会综合融资成本。强化货币\n政策执行和监督，完善利率自律管理。三是加大对重大战略、重点领\n域和薄弱环节的支持力度。丰富完善结构性货币政策工具体系，调整\n优化信贷结构，支持做好金融“五篇大文章”。增加科技创新和技术\n改造再贷款、支农支小再贷款额度各3000 亿元，创设5000 亿元服务\n消费与养老再贷款、2000 亿元科技创新债券风险分担工具。四是保\n持汇率基本稳定。坚持市场在汇率形成中起决定性作用，发挥好汇率\n对宏观经济、国际收支的调节功能，综合施策，保持人民币汇率在合\n理均衡水平上的基本稳定。五是重点领域金融风险持续收敛。整合设\n立中国人民银行宏观审慎和金融稳定委员会，进一步健全宏观审慎管\n\nII\n理和金融稳定保障体系。优化支持资本市场两项货币政策工具，支持\n发挥汇金公司“类平准基金”作用。稳步推进重点机构和重点区域金\n融风险处置。\n2025 年适度宽松的货币政策效果逐'

## 4. 文本清洗和章节识别

In [6]:
from src.monetary_policy.text.text_cleaner import normalize_text, split_sentences
cleaned = normalize_text(raw_text[:1200])
print(cleaned[:500])
print('sentences:', len(split_sentences(cleaned)))

中国货币政策执行报告 2025 年第四季度 中国人民银行货币政策分析小组 2026 年2 月10 日 I 内容摘要 2025 年是“十四五”收官之年，在以习近平同志为核心的党中 央坚强领导下，国民经济延续稳中有进发展态势，经济社会发展主要 目标顺利实现。
全年国内生产总值（GDP）同比增长5%。
中国人民银 行坚持以习近平新时代中国特色社会主义思想为指导，坚决落实党中 央、国务院决策部署，实施适度宽松的货币政策，在执行好存量货币 政策的基础上，又推出一揽子货币金融政策组合，强化逆周期调节， 有效支持实体经济稳定增长和金融市场平稳运行。
 一是保持货币信贷合理增长。
综合运用存款准备金率、公开市场 操作等多种货币政策工具，保持流动性充裕。
引导金融机构加强项目 储备和信贷投放，充分满足实体经济有效信贷需求。
二是推动社会综 合融资成本低位下行。
下调政策利率、结构性货币政策工具利率和个 人住房公积金贷款利率，有力支持降低社会综合融资成本。
强化货币 政策执行和监督，完善利率自律管理。
三是加大对重大战略、重点领 域和薄弱环节的支持力度。
丰富完善结构性货币政策工具体系，调整 优化信贷结构
sentences: 25


## 5. 早期政策指引章节修复

In [7]:
from src.monetary_policy.text.section_repair import repair_guidance_sections
repair_guidance_sections()
repair = pd.read_excel(ROOT / 'output/diagnostics/section_repair_report.xlsx')
repair

,report_id,old_found,old_char_count,new_found,new_char_count,extraction_rule,manual_override,review_note
0,2006Q1,False,0,True,1914,manual_title_alias_last_match,True,正文第五部分第二节标题，目录中同名标题不使用；取最后一次匹配。
1,2006Q4,False,0,True,1701,manual_title_alias_last_match,True,正文第五部分第二节标题，取最后一次匹配。
2,2007Q2,False,0,True,1072,manual_title_alias_last_match,True,正文第五部分第二节标题，取最后一次匹配。
3,2007Q4,False,0,True,2252,manual_title_alias_last_match,True,正文第五部分第三节标题，取最后一次匹配。


In [8]:
sections = pd.read_csv(ROOT / 'data/processed/report_sections_repaired.csv')
sections[(sections['report_id'].isin(['2006Q1','2006Q4','2007Q2','2007Q4'])) & (sections['section']=='guidance')][['report_id','found','char_count','local_path']]

,report_id,found,char_count,local_path
0,2006Q1,True,1914,data\interim\report_sections\2006Q1_guidance.txt
9,2006Q4,True,1701,data\interim\report_sections\2006Q4_guidance.txt
15,2007Q2,True,1072,data\interim\report_sections\2007Q2_guidance.txt
21,2007Q4,True,2252,data\interim\report_sections\2007Q4_guidance.txt


## 6. 中文分词和自定义政策短语

In [9]:
from src.monetary_policy.text.tokenizer import tokenize
example = '保持流动性合理充裕，不搞大水漫灌，强化逆周期调节和跨周期调节。'
tokenize(example)

Building prefix dict from the default dictionary ...


Loading model from cache C:\Users\35494\AppData\Local\Temp\jieba.cache


Loading model cost 1.675 seconds.


Prefix dict has been built successfully.


['保持', '流动性合理充裕', '不搞大水漫灌', '强化', '逆周期调节', '跨周期调节']

## 7. 金融情感词典和 PBC 领域词典

In [10]:
from src.monetary_policy.text.lexicon import build_combined_lexicon, COMBINED_PATH
lexicon = build_combined_lexicon()
print(len(lexicon.positive), len(lexicon.negative), len(lexicon.dovish), len(lexicon.hawkish))
pd.read_csv(COMBINED_PATH).head()

4714 7236 35 33


,word,category
0,一专多能,positive
1,一丝不苟,positive
2,一举,positive
3,一举两得,positive
4,一举多得,positive


## 8. 否定词和程度副词处理示例

In [11]:
from src.monetary_policy.text.sentiment import score_text
for s in ['加大支持实体经济力度。','不搞大水漫灌。','更加有力支持稳增长。']:
    print(s, score_text(s, lexicon))

加大支持实体经济力度。 {'raw_positive_count': 1.0, 'raw_negative_count': 0.0, 'raw_dovish_count': 2.0, 'raw_hawkish_count': 0.0, 'normalized_sentiment': 90.9090909090909, 'normalized_policy_stance': 181.8181818181818, 'effective_chars': 11, 'sentence_count': 1, 'attention_growth': 90.9090909090909, 'attention_inflation': 0.0, 'attention_risk': 0.0, 'attention_exchange_rate': 0.0, 'attention_financial_stability': 0.0}
不搞大水漫灌。 {'raw_positive_count': 0.0, 'raw_negative_count': 0.0, 'raw_dovish_count': 0.0, 'raw_hawkish_count': 1.0, 'normalized_sentiment': 0.0, 'normalized_policy_stance': -142.85714285714286, 'effective_chars': 7, 'sentence_count': 1, 'attention_growth': 0.0, 'attention_inflation': 0.0, 'attention_risk': 0.0, 'attention_exchange_rate': 0.0, 'attention_financial_stability': 0.0}
更加有力支持稳增长。 {'raw_positive_count': 3.864, 'raw_negative_count': 0.0, 'raw_dovish_count': 2.184, 'raw_hawkish_count': 0.0, 'normalized_sentiment': 386.4, 'normalized_policy_stance': 218.4, 'effective_chars': 10,

## 9. 文本指标计算

In [12]:
from src.monetary_policy.pipeline import build_text_features
features = build_text_features()
features[['report_id','guidance_z_sentiment','macro_z_sentiment','guidance_z_policy_stance','guidance_unexpected_tone']].tail()

,report_id,guidance_z_sentiment,macro_z_sentiment,guidance_z_policy_stance,guidance_unexpected_tone
76,2025Q1,1.025574,1.093856,0.636252,0.023255
77,2025Q2,1.487339,1.419508,1.510001,1.059508
78,2025Q3,1.096325,1.324520,0.492275,-0.602210
79,2025Q4,1.548600,1.387774,0.386906,0.032906
80,2026Q1,0.134750,1.592605,-0.215035,-0.495478


## 10. 扩展 TF-IDF 创新度

In [13]:
features[['report_id','in_formal_sample','guidance_similarity_expanding_tfidf','guidance_novelty','fulltext_novelty_expanding_tfidf','similarity_char_ngram']].dropna(subset=['guidance_novelty']).tail()

,report_id,in_formal_sample,guidance_similarity_expanding_tfidf,guidance_novelty,fulltext_novelty_expanding_tfidf,similarity_char_ngram
76,2025Q1,True,0.815826,0.184174,0.093828,0.957835
77,2025Q2,True,0.784184,0.215816,0.099886,0.960640
78,2025Q3,True,0.786610,0.213390,0.109070,0.942939
79,2025Q4,True,0.839593,0.160407,0.103659,0.941472
80,2026Q1,False,0.804280,0.195720,0.115702,NaN


## 11. 主题关注和未预期语调

In [14]:
features.loc[features['in_formal_sample'], ['publication_datetime','guidance_z_sentiment','macro_z_sentiment','guidance_z_policy_stance','guidance_attention_growth','guidance_attention_inflation','guidance_unexpected_tone']].describe()

,publication_datetime,guidance_z_sentiment,macro_z_sentiment,guidance_z_policy_stance,guidance_attention_growth,guidance_attention_inflation,guidance_unexpected_tone
count,80,8.000000e+01,8.000000e+01,8.000000e+01,80.000000,80.000000,79.000000
mean,2016-03-27 18:55:06.825000,-3.441691e-16,-2.220446e-16,8.881784e-17,13.031454,1.007921,0.158862
min,2006-05-31 14:12:00,-2.679498e+00,-3.042341e+00,-1.972799e+00,6.170599,0.000000,-1.640146
25%,2011-04-10 12:47:24.750000,-7.103155e-01,-7.583922e-01,-6.989173e-01,9.209940,0.323185,-0.370914
50%,2016-03-22 17:04:04.500000,-9.734486e-02,-1.341854e-02,-2.079365e-02,12.272538,0.834185,0.158628
75%,2021-03-03 21:29:38.750000,6.017127e-01,8.976911e-01,6.669137e-01,15.305887,1.409074,0.590154
max,2026-02-10 19:23:18,3.085042e+00,1.678018e+00,2.869649e+00,38.048780,5.947324,2.969713
std,NaN,1.006309e+00,1.006309e+00,1.006309e+00,5.514848,1.046043,0.751873


## 12. 人工标注完成后的文本验证

In [15]:
from src.monetary_policy.text.manual_validation import load_filled_annotations, has_filled_annotations
print('人工标注已完成:', has_filled_annotations())
filled = load_filled_annotations()
print('标注句子数:', len(filled))
print('标注人:', filled['reviewer'].unique())
print()
print('情感标签分布:')
print(filled['manual_sentiment_label'].value_counts())
print()
print('政策倾向标签分布:')
print(filled['manual_policy_stance_label'].value_counts())
print()
print('主题标签分布:')
print(filled['manual_topic_label'].value_counts())

人工标注已完成: True
标注句子数: 240
标注人: <StringArray>
['罗允绩']
Length: 1, dtype: str

情感标签分布:
manual_sentiment_label
neutral     176
positive     51
negative     13
Name: count, dtype: int64

政策倾向标签分布:
manual_policy_stance_label
irrelevant    174
neutral        30
dovish         26
hawkish        10
Name: count, dtype: int64

主题标签分布:
manual_topic_label
growth                 166
other                   28
financial_stability     17
risk                    12
inflation                9
exchange_rate            7
real_estate              1
Name: count, dtype: int64


In [16]:
from src.monetary_policy.text.validation_report import run_text_validation
vresult = run_text_validation()
print(f"情感 Accuracy: {vresult['summary']['sentiment_accuracy']:.4f}")
print(f"情感 Macro-F1: {vresult['summary']['sentiment_macro_f1']:.4f}")
print(f"政策倾向 Accuracy: {vresult['summary']['stance_accuracy']:.4f}")
print(f"政策倾向 Macro-F1: {vresult['summary']['stance_macro_f1']:.4f}")
print(f"主题 Accuracy: {vresult['summary']['topic_accuracy']:.4f}")
print(f"主题 Macro-F1: {vresult['summary']['topic_macro_f1']:.4f}")
print()
print('不一致句子总数:', vresult['summary']['disagreement_count'])
print('  情感不一致:', vresult['summary']['disagreement_sentiment_count'])
print('  政策倾向不一致:', vresult['summary']['disagreement_stance_count'])
print('  主题不一致:', vresult['summary']['disagreement_topic_count'])

情感 Accuracy: 0.2958
情感 Macro-F1: 0.3272
政策倾向 Accuracy: 0.1792
政策倾向 Macro-F1: 0.2681
主题 Accuracy: 0.6583
主题 Macro-F1: 0.5532

不一致句子总数: 239
  情感不一致: 169
  政策倾向不一致: 197
  主题不一致: 82


## 13. 股票数据清洗

In [17]:
stock = pd.read_csv(ROOT / 'data/processed/csi300_daily.csv', parse_dates=['date'])
stock[['date','close','simple_return','volatility_20d']].tail()

,date,close,simple_return,volatility_20d
5926,2026-06-12,4777.321,0.011627,0.011598
5927,2026-06-15,4891.713,0.023945,0.012781
5928,2026-06-16,4884.232,-0.001529,0.012765
5929,2026-06-17,4931.386,0.009654,0.012928
5930,2026-06-18,4941.599,0.002071,0.012447


## 14. 债券收益率曲线数据

In [18]:
bond = pd.read_csv(ROOT / 'data/processed/government_bond_yields.csv', parse_dates=['date'])
bond[['date','yield_1y','yield_5y','yield_10y','spread_10y_1y']].tail()

,date,yield_1y,yield_5y,yield_10y,spread_10y_1y
5072,2026-06-12,1.1967,1.4776,1.7427,0.5460
5073,2026-06-15,1.2017,1.4832,1.7446,0.5429
5074,2026-06-16,1.1967,1.4573,1.7324,0.5357
5075,2026-06-17,1.1892,1.4529,1.7270,0.5378
5076,2026-06-18,1.1717,1.4407,1.7299,0.5582


## 15. 发布时间和交易日对齐

In [19]:
events = pd.read_csv(ROOT / 'data/processed/event_calendar.csv')
events['in_formal_sample'] = events['report_period'].map(is_in_formal_sample)
events['action_nearby_core'] = events['action_nearby']
events['action_nearby_extended'] = events.get('action_nearby_extended', events['action_nearby'])
events.loc[events['in_formal_sample'], ['event_id','publication_datetime','bond_event_date','equity_event_date','action_nearby_core','action_nearby_extended']].tail()

,event_id,publication_datetime,bond_event_date,equity_event_date,action_nearby_core,action_nearby_extended
75,2024Q4,2025-02-13 17:05:21,2025-02-14,2025-02-14,0,0
76,2025Q1,2025-05-09 17:40:30,2025-05-12,2025-05-12,1,1
77,2025Q2,2025-08-15 18:00:29,2025-08-18,2025-08-18,0,0
78,2025Q3,2025-11-11 17:00:27,2025-11-12,2025-11-12,0,0
79,2025Q4,2026-02-10 19:23:18,2026-02-11,2026-02-11,0,0


## 16. 修正后的窗口收益函数

In [20]:
from src.monetary_policy.events.event_windows import window_return
prices = pd.Series([100, 102, 105, 110, 121])
print('0 to +3:', window_return(prices, 1, 0, 3))
print('-1 to +1:', window_return(prices, 1, -1, 1))

0 to +3: 0.18627450980392157
-1 to +1: 0.050000000000000044


## 17. 事件面板

In [21]:
from src.monetary_policy.events.event_panel import build_stock_event_panel, build_yield_curve_event_panel
stock_panel = build_stock_event_panel(features)
curve_daily, curve_panel = build_yield_curve_event_panel(features)
stock_panel[['event_id','return_0_p3','rv_0_5','log_rv_0_5','pre_event_volatility_20d']].tail()

,event_id,return_0_p3,rv_0_5,log_rv_0_5,pre_event_volatility_20d
75,2024Q4,0.000292,0.125804,-2.073027,0.009014
76,2025Q1,0.004264,0.139112,-1.972477,0.004322
77,2025Q2,0.011478,0.153496,-1.874082,0.006295
78,2025Q3,-0.010300,0.153627,-1.873230,0.010488
79,2025Q4,-0.001331,0.124125,-2.086470,0.008039


## 18. 描述性统计

In [22]:
stock_panel[['log_rv_0_5','guidance_novelty','guidance_novelty_x_post_2019','return_0_p3','guidance_z_sentiment']].describe()

,log_rv_0_5,guidance_novelty,guidance_novelty_x_post_2019,return_0_p3,guidance_z_sentiment
count,80.000000,79.000000,79.000000,80.000000,8.000000e+01
mean,-1.705805,0.278383,0.084280,0.002683,-3.441691e-16
std,0.496365,0.163187,0.126103,0.023513,1.006309e+00
min,-2.890303,0.045684,0.000000,-0.070100,-2.679498e+00
25%,-2.008664,0.175238,0.000000,-0.010619,-7.103155e-01
50%,-1.741285,0.217234,0.000000,-0.000010,-9.734486e-02
75%,-1.426342,0.342516,0.192829,0.017391,6.017127e-01
max,-0.301078,0.742982,0.590192,0.073727,3.085042e+00


## 19. 股票波动率主结果

In [23]:
from src.monetary_policy.analysis.stock_volatility import run_stock_volatility_models
vol_table, main_vol, egarch = run_stock_volatility_models(stock_panel)
vol_table

,model,dependent,target,n,beta,se_hc3,p_value,r2,effect_percent_if_log_y
0,full,log_rv_0_5,guidance_novelty,79,0.746645,0.334847,0.025760,0.454702,110.990945
1,pre_2019,log_rv_0_5,guidance_novelty,50,0.049356,0.397946,0.901295,0.462765,5.059393
2,post_2019,log_rv_0_5,guidance_novelty,29,1.489204,1.435850,0.299662,0.242862,343.356437
3,covid,log_rv_0_5,guidance_novelty,12,-0.489641,6.494565,0.939902,0.160210,-38.715385
4,non_covid,log_rv_0_5,guidance_novelty,67,0.474270,0.364760,0.193524,0.493083,60.684087


In [24]:
main_vol['params'], main_vol['pvalues'], main_vol['post_2019_total_effect'], main_vol['economic_effect']

({'const': -2.260683276860103,
  'guidance_novelty': 0.7466450326474896,
  'pre_event_volatility_20d': 28.74312337199772,
  'action_nearby_core': 0.07872414030132373,
  'post_2019': -0.1429931494862665,
  'guidance_novelty_x_post_2019': -0.4556596161865616},
 {'const': 2.778938119540168e-57,
  'guidance_novelty': 0.025760080236270067,
  'pre_event_volatility_20d': 0.00015493133005706065,
  'action_nearby_core': 0.3464351930951248,
  'post_2019': 0.5290190599336522,
  'guidance_novelty_x_post_2019': 0.5011016272718196},
 {'estimate': 0.290985416460928,
  'se_hc3': 0.6466179847624756,
  'p_value': 0.6527022742493203,
  'conf_int_95': [-0.976385833673524, 1.55835666659538]},
 {'one_unit_guidance_novelty_log_rv_beta': 0.7466450326474896,
  'one_unit_guidance_novelty_percent_change_in_rv': np.float64(110.9909451629524)})

## 20. 股票收益结果

In [25]:
from src.monetary_policy.analysis.stock_returns import run_stock_return_models
return_table = run_stock_return_models(stock_panel)
return_table.head(12)

,model,dependent,target,n,beta,se_hc3,p_value,r2,effect_percent_if_log_y
0,guidance_financial_sentiment,return_0_p1,guidance_z_sentiment,80,-0.000399,0.002084,0.848299,0.006280,-0.039855
1,macro_financial_sentiment,return_0_p1,macro_z_sentiment,80,-0.000154,0.001582,0.922513,0.005541,-0.015385
2,policy_stance,return_0_p1,guidance_z_policy_stance,80,-0.000759,0.001293,0.556919,0.008578,-0.075918
3,unexpected_tone,return_0_p1,guidance_unexpected_tone,79,0.000187,0.002960,0.949684,0.004539,0.018678
4,guidance_financial_sentiment,return_0_p3,guidance_z_sentiment,80,0.001072,0.003624,0.767300,0.017501,0.107289
5,macro_financial_sentiment,return_0_p3,macro_z_sentiment,80,0.000982,0.002640,0.710005,0.017170,0.098229
6,policy_stance,return_0_p3,guidance_z_policy_stance,80,-0.002197,0.002177,0.312943,0.024230,-0.219448
7,unexpected_tone,return_0_p3,guidance_unexpected_tone,79,-0.003765,0.003411,0.269739,0.028916,-0.375746
8,guidance_financial_sentiment,return_m1_p1,guidance_z_sentiment,80,0.000087,0.003237,0.978669,0.014241,0.008655
9,macro_financial_sentiment,return_m1_p1,macro_z_sentiment,80,0.004581,0.002214,0.038567,0.060031,0.459114


## 21. 收益率曲线水平、斜率和曲率

In [26]:
curve_daily[['date','level','slope','curvature']].tail()

,date,level,slope,curvature
5072,2026-06-12,1.472333,0.5460,0.0158
5073,2026-06-15,1.476500,0.5429,0.0201
5074,2026-06-16,1.462133,0.5357,-0.0145
5075,2026-06-17,1.456367,0.5378,-0.0104
5076,2026-06-18,1.447433,0.5582,-0.0202


In [27]:
from src.monetary_policy.analysis.yield_curve import run_yield_curve_models
yield_table = run_yield_curve_models(curve_panel)
yield_table

,model,dependent,target,n,beta,se_hc3,p_value,r2,effect_percent_if_log_y,post_2019_interaction_beta,post_2019_total_effect,post_2019_total_p_value,bootstrap_ci_95_base,bootstrap_ci_95_total,permutation_p_base,permutation_p_interaction
0,main_yield_curve,delta_slope_bp_0_3,guidance_unexpected_tone,79,0.954200,1.185767,0.420987,0.058078,159.659209,-1.562208,-0.608008,0.723921,"[-0.9267454017622101, 3.654858620795652]","[-3.779766468098385, 1.2275147245224713]",0.383085,0.333333
1,secondary_yield_curve,delta_level_bp_0_3,guidance_unexpected_tone,79,0.541721,0.894063,0.544575,0.021241,71.896307,-0.970190,-0.428469,0.727257,"[-1.4856536651811287, 1.997540007238475]","[-3.223456656331668, 0.6077964970082458]",0.452736,0.407960
2,secondary_yield_curve,delta_curvature_bp_0_3,guidance_unexpected_tone,79,1.723496,1.493032,0.248353,0.085852,460.408352,-3.954519,-2.231024,0.018532,"[-1.3429637048100345, 3.964373069691203]","[-4.7313966657514115, 0.562955542773198]",0.074627,0.009950
3,legacy_1y_m1_p3,delta_yield_1y_bp_m1_p3,guidance_tone_change,80,-1.080867,1.912015,0.571868,0.031762,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 22. 原债券规格对照

In [28]:
legacy = json.loads((ROOT / 'output/results/legacy_primary_result.json').read_text(encoding='utf-8')) if (ROOT / 'output/results/legacy_primary_result.json').exists() else json.loads((ROOT / 'output/results/primary/PRIMARY_RESULT_LOCK.json').read_text(encoding='utf-8'))
legacy['n'], legacy['params']['guidance_tone_change'], legacy['pvalues']['guidance_tone_change']

(80, -1.0808667061037642, 0.5718680236295097)

## 23. 稳健性检验和 Holm 校正

In [29]:
from src.monetary_policy.analysis.robustness import similarity_robustness
similarity_robustness(stock_panel)

,model,dependent,target,n,beta,se_hc3,p_value,r2,effect_percent_if_log_y,p_value_holm
0,similarity_robustness,log_rv_0_5,guidance_novelty,79,0.746645,0.334847,0.025760,0.454702,110.990945,0.068143
1,similarity_robustness,log_rv_0_5,fulltext_novelty_expanding_tfidf,79,8.439640,3.421143,0.013629,0.463443,462589.031587,0.068143
2,similarity_robustness,log_rv_0_5,guidance_novelty_full_sample_tfidf,79,0.704353,0.289925,0.015122,0.459743,102.253693,0.068143
3,similarity_robustness,log_rv_0_5,fulltext_novelty_full_sample_tfidf,79,5.700157,2.604500,0.028627,0.465603,29791.431247,0.068143
4,similarity_robustness,log_rv_0_5,z_similarity_char_ngram,79,-0.184880,0.078858,0.019055,0.470628,-16.879570,0.068143


## 24. 图表源数据

In [30]:
sorted([p.name for p in (ROOT / 'output/figures').glob('figure*.png')])

['figure1_tone_series.png',
 'figure2_similarity.png',
 'figure3_volatility_paths.png',
 'figure4_similarity_rv_scatter.png',
 'figure5_yield_curve_factors.png',
 'figure6_curve_reactions.png',
 'figure_policy_confusion_matrix.png',
 'figure_policy_direction_confusion_matrix.png',
 'figure_sentiment_confusion_matrix.png',
 'figure_topic_confusion_matrix.png']

## 25. 结论和局限

文本创新度、股票收益和收益率曲线结果均由上述模块现场计算。解释时只讨论相关性，不把事件研究回归写成因果识别。

### 文本验证结论

人工标注已完成（标注人：罗允绩，240 句，涵盖政策指引和宏观章节）。验证结论如下：

- **金融情感**：自动词典的句子级准确率为 26.25%，主要问题是过度预测 positive（157/184 人工标注 neutral 的句子被自动分类为 positive）。原因在于姜富伟等和 Du et al. 的中文金融情感词典面向文档级金融市场分析设计，直接用于句子级政策文本时会产生系统性正向偏误。文档级聚合后的标准化指标在回归中更为稳健。
- **政策倾向**：自动词典的句子级准确率为 14.17%，hawkish 召回率为 0%。PBC 领域鹰鸽词典（v2 已扩展至 35+33 词）在句子级仍存在严重覆盖不足，原因是政策倾向常通过隐含、间接表达传递。文档级聚合可在一定程度上缓解这一问题。
- **主题分类**：v2 主题词典扩展后准确率从 38.33% 提升至 58.75%，growth 召回率从 35.1% 提升至 62.3%，financial_stability 召回率从 0% 提升至 36.8%。但 risk 和 inflation 的召回率仍偏低。

词典修订版本已保存在 `data/dictionaries/lexicon_versions/` 目录下，含 v1→v2 变更报告。